<div style="display:flex; align-items:center; gap:18px; text-align:left">
<img src="https://sebastiancontz.github.io/ust-introduccion-machine-learning/assets/logo-ust.svg" width="100" alt="Logo de la Universidad Santo Tomás">
<div>
<p>Ingeniería en Información y Control de Gestión</p>
<p>Facultad de Economía y Negocios</p>
<p>Introducción a Machine Learning</p>
<p>Semana 05: Ingeniería de variables y preprocesamiento</p>
</div>
</div>

# 05 · Sesión 4, parte B — representar lo que ya está bien

En la **parte A de esta misma sesión** corregimos lo que estaba mal en el archivo. Acá
**transformamos** lo que ya está bien, para que el modelo pueda aprovecharlo.

El archivo es el mismo, con las siete decisiones de la parte A ya aplicadas. Y la **bitácora sigue
abierta**: no se abre otra. Si la decisión 6 de la parte A quedó pendiente, se resuelve en su
cuaderno, no acá.

## Qué vamos a hacer

1. Construir **un preprocesador** que trate por separado las columnas numéricas y las categóricas.
2. Ver **dónde el one-hot deja de servir**, con el código de producto.
3. **Crear una variable nueva** justificada desde el negocio, incorporarla y medirla.

## Criterio de éxito de la actividad

La justificación de la variable que creen explica **de dónde sale su valor** y **a qué hecho del negocio
corresponde**, sin apelar a cuánto bajó el error.

## Preparación del entorno

In [1]:
%%capture
!pip install -q pandas numpy pyarrow scikit-learn

In [2]:
import os

import numpy as np
import pandas as pd

def num(x, dec=0):
    """Miles con punto y decimales con coma, como en el resto del material del curso."""
    return f'{x:,.{dec}f}'.replace(',', '@').replace('.', ',').replace('@', '.')

## Cargar datos

El archivo viene en **Parquet**, que guarda el tipo de cada columna, así que no hay que adivinar nada al
leerlo. Ya usamos `read_parquet` en la clase 4.

In [3]:
url_datasets = 'https://raw.githubusercontent.com/sebastiancontz/ust-introduccion-machine-learning-colab/main/ediciones/2026/datasets/'
ruta_datos = '../datasets/' if os.path.exists('../datasets') else url_datasets
ventas = pd.read_parquet(ruta_datos + 'ventas_online_limpio.parquet')

In [4]:
print(f'El archivo trae {num(len(ventas))} registros y {ventas.shape[1]} columnas.')

El archivo trae 536.642 registros y 14 columnas.


Cada **registro** es una **línea de factura**: un producto dentro de una factura. No es una venta completa y
no es un cliente.

### El filtro que fija la base, y se declara una sola vez

Nos quedamos con las líneas que son **ventas de producto efectivas**: que sean producto, que no estén
canceladas, y que tengan cantidad y precio positivos. Todas las cifras de este cuaderno salen de ahí.

<!-- contrato: accion=fijar-base -->

In [5]:
lineas = ventas[
    ventas["es_producto"] & ~ventas["cancelada"] & (ventas["cantidad"] > 0) & (ventas["precio_unitario"] > 0)
].copy()
lineas["monto_linea"] = lineas["cantidad"] * lineas["precio_unitario"]   # lo que vale esa línea

In [6]:
print(f'Quedan {num(len(lineas))} líneas de las {num(len(ventas))} originales.')

Quedan 522.504 líneas de las 536.642 originales.


## Y ahora cada registro va a ser una factura

Para lo que sigue necesitamos **un registro por factura**, no por línea. O sea que cambiamos la **unidad de
observación**, igual que en la clase 2, y por eso lo decimos en voz alta.

### `agg`: resumir cada grupo con la función que corresponda

`groupby` ya lo conocen: separa las filas en grupos. `agg` es lo que va después: recibe, por cada columna
nueva, **de dónde sale y cómo se resume**. El monto de una factura es la **suma** de sus líneas, su
número de líneas es un **conteo**, y su país es el **primero** —todas sus líneas comparten país—.

El **mes** no viene como columna: se saca de la fecha con el accesor `.dt`, que da acceso a las partes
de una fecha —año, mes, día, hora—. Es la familia de variables derivadas «componentes de una fecha».

<!-- contrato: accion=agregar-por-factura -->

In [7]:
facturas = lineas.groupby('n_documento').agg(
    monto_total=('monto_linea', 'sum'),
    n_lineas=('codigo_producto', 'size'),
    unidades=('cantidad', 'sum'),
    pais=('pais', 'first'),
    fecha=('fecha_factura', 'first'),
).reset_index()

In [8]:
facturas["mes"] = facturas["fecha"].dt.month   # el mes sale de la fecha, no viene como columna

In [9]:
print(f'De {num(len(lineas))} líneas quedaron {num(len(facturas))} facturas.')

De 522.504 líneas quedaron 19.773 facturas.


<p align="center"></p>

## Cómo vamos a medir

Necesitamos una forma de comparar dos maneras de escribir las mismas variables. Vamos a usar un **modelo
lineal** —el que ya conocen— como **instrumento de medición**, no como predictor: explica el monto de una
factura desde su número de líneas, sus unidades, su país y su mes.

Y lo vamos a reportar en **libras por factura**: la mitad de las facturas queda estimada con un error
menor a esa cifra. Nada de porcentajes abstractos.

### Dos detalles de la receta, y los dos importan

- El modelo trabaja sobre el **logaritmo** del monto, porque su distribución tiene una cola larguísima.
  Así que la predicción hay que **destransformarla** con `expm1` antes de compararla con libras reales.
- `log1p` calcula el logaritmo de **1 + x**, no de x. El logaritmo de cero no existe, y este archivo
  tiene facturas de una sola unidad.

### Dos nombres nuevos, y uno ya lo vieron en las slides

`make_pipeline` encadena pasos: le entregamos el preprocesador y el modelo, y devuelve un solo objeto
que aplica el primero y después el segundo. Es la forma corta de armar el **pipeline** del que hablamos
en clase.

Y una advertencia sobre la cifra: **se calcula sobre las mismas facturas con las que se ajustó el
modelo**. Sirve para comparar dos formas de escribir las mismas variables, que es lo único que
necesitamos hoy. Por qué eso no alcanza para saber si un modelo sirve es la clase 6.

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

def error_libras(preprocesador, columnas):
    """Mediana del error absoluto, en libras por factura."""
    modelo = make_pipeline(preprocesador, LinearRegression())
    modelo.fit(facturas[columnas], np.log1p(facturas["monto_total"]))
    prediccion = np.expm1(modelo.predict(facturas[columnas]))
    return float(np.median(np.abs(prediccion - facturas["monto_total"])))

In [11]:
print('Referencia de lectura: la factura mediana vale '
      f'{num(facturas["monto_total"].median())} libras.')

Referencia de lectura: la factura mediana vale 302 libras.


# Primer tiempo · el preprocesador mínimo

Cuatro columnas entran: dos numéricas y dos categóricas. Cada tipo necesita algo distinto.

### Las tres piezas que hacen falta

- **`OneHotEncoder`** convierte una columna de categorías en una columna por categoría. Con
  `handle_unknown='ignore'` no se cae si mañana aparece un país que no estaba.
- **`FunctionTransformer`** aplica una función cualquiera a las columnas que se le indiquen; acá le
  vamos a pasar `log1p`.
- **`ColumnTransformer`** es el que reparte: recibe una lista de tripletas —nombre, transformador,
  columnas— y reúne todas las salidas en una sola matriz.

### Y por qué las numéricas pasan por el logaritmo

Porque es la decisión que la teoría de hoy justificó, y conviene no ejecutarla a ciegas: `unidades`
tiene una cola larguísima, y su máximo son **80.995 unidades en una sola factura** — la misma que la
clase 4 discutió y decidió **conservar**, porque era una venta real con su devolución.

Transformar la variable es lo que desarma esa cola **sin borrar la venta**. Acá no lo volvemos a
demostrar: se aplica la decisión y se sigue.

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

columnas_numericas = ['n_lineas', 'unidades']
columnas_categoricas = ['pais', 'mes']

In [13]:
preprocesador = ColumnTransformer([
    ('num', FunctionTransformer(np.log1p), columnas_numericas),
    ('cat', OneHotEncoder(handle_unknown='ignore'), columnas_categoricas),
])

Hasta acá solo lo **declaramos**. Ahora lo ejecutamos sobre el archivo y miramos qué sale.

<!-- contrato: cifra=52 -->

In [14]:
matriz = preprocesador.fit_transform(facturas[columnas_numericas + columnas_categoricas])
print(f'Entraron {len(columnas_numericas + columnas_categoricas)} columnas y salieron {matriz.shape[1]}.')

Entraron 4 columnas y salieron 52.


### De dónde salen esas columnas

Las dos numéricas siguen siendo dos. Las categóricas son las que se multiplicaron: una columna por cada
valor distinto.

In [15]:
for columna in columnas_categoricas:
    print(f'{columna:8s} {facturas[columna].nunique():>3} valores distintos')

pais      38 valores distintos
mes       12 valores distintos


### Y casi todo lo que se agregó son ceros

In [16]:
# la matriz viene comprimida, sin guardar los ceros; la expandimos solo para contarlos
densa = matriz.toarray() if hasattr(matriz, 'toarray') else matriz
print(f'El {num(100 * (densa == 0).mean())} % de esa matriz son ceros.')

El 92 % de esa matriz son ceros.


<p align="center"></p>

### El error de este preprocesador

Guardemos la cifra, porque es la vara contra la que vamos a comparar todo lo demás.

In [17]:
error_base = error_libras(preprocesador, columnas_numericas + columnas_categoricas)
print(f'Error mediano: {num(error_base, 1)} libras por factura.')

Error mediano: 71,2 libras por factura.


# Segundo tiempo · dónde el one-hot deja de servir

Probemos con el **código de producto**. Y para eso hay que **volver a la tabla de líneas**, porque el
código de producto solo existe ahí: una factura tiene muchos productos, no un código.

In [18]:
n_productos = lineas["codigo_producto"].nunique()
print(f'Hay {num(n_productos)} productos distintos en {num(len(lineas))} líneas.')

Hay 3.900 productos distintos en 522.504 líneas.


### Antes de ejecutar nada, hagamos la cuenta

Si le aplicáramos el one-hot, tendríamos una columna por producto y un registro por línea.

In [19]:
celdas = len(lineas) * n_productos
gigas = celdas * 8 / 1e9   # 8 bytes por número en punto flotante
print(f'Serían {num(celdas / 1e6)} millones de celdas, '
      f'o {num(gigas, 1)} GB si se guardaran todas.')

Serían 2.038 millones de celdas, o 16,3 GB si se guardaran todas.


In [20]:
print(f'Y solo el {num(100 * len(lineas) / celdas, 3)} % sería distinto de cero.')

Y solo el 0,026 % sería distinto de cero.


### Qué concluimos

Dos cosas, y las dos importan:

1. Acá **la herramienta correcta no es esta**. Hay formas de guardar solo las celdas que no son cero, y
   con eso el archivo entra en memoria — pero el problema no era el espacio: el modelo seguiría teniendo
   que estimar un número por cada producto.
2. El código de producto **no puede entrar como una columna más** de la tabla por factura. Por eso hubo
   que agregar al principio. La agregación no fue una preferencia: fue la consecuencia de esto.

# Tercer tiempo · ahora ustedes

Toca **crear una variable nueva** que el archivo no trae, incorporarla al preprocesador y medirla.

## La candidata: el precio de catálogo medio de la canasta

La idea de negocio es simple: **una factura de productos caros vale más que una de productos baratos**,
aunque las dos tengan la misma cantidad de líneas y de unidades. Eso el archivo no lo dice directamente,
pero se puede construir.

Dos pasos:

1. Para cada producto, su **precio habitual**: la mediana de todos los precios a los que se vendió. Sale
   con el mismo `agg` de antes, solo que agrupando por producto en vez de por factura.
2. Para cada factura, el **promedio de esos precios habituales** entre los productos que lleva.

### `merge`: pegarle a cada línea un dato que vive en otra tabla

`merge` une dos tablas por una columna que comparten. Acá la tabla chica es el catálogo de precios
habituales, y la columna compartida es `codigo_producto`.

In [21]:
catalogo = lineas.groupby('codigo_producto').agg(
    precio_catalogo=('precio_unitario', 'median'),
).reset_index()

In [22]:
catalogo.head()

,codigo_producto,precio_catalogo
0,10002,0.85
1,10080,0.39
2,10120,0.21
3,10123C,0.65
4,10124A,0.42


Ahora se lo pegamos a cada línea, y de ahí lo resumimos por factura.

<!-- contrato: accion=pegar-catalogo -->

In [23]:
lineas_con_catalogo = lineas.merge(catalogo, on='codigo_producto', how='left')

In [24]:
por_factura = lineas_con_catalogo.groupby("n_documento")["precio_catalogo"].mean()
facturas["precio_catalogo_medio"] = facturas["n_documento"].map(por_factura)

In [25]:
print(f'La variable nueva va de {num(facturas["precio_catalogo_medio"].min(), 2)} '
      f'a {num(facturas["precio_catalogo_medio"].max())} libras, '
      f'con mediana {num(facturas["precio_catalogo_medio"].median(), 2)}.')

La variable nueva va de 0,04 a 165 libras, con mediana 2,78.


## Incorporarla al preprocesador

Va por la **ruta numérica**, junto a las otras dos, así que también pasa por el logaritmo.

In [26]:
columnas_numericas_con_precio = columnas_numericas + ['precio_catalogo_medio']
preprocesador_nuevo = ColumnTransformer([
    ('num', FunctionTransformer(np.log1p), columnas_numericas_con_precio),
    ('cat', OneHotEncoder(handle_unknown='ignore'), columnas_categoricas),
])

In [27]:
error_nuevo = error_libras(preprocesador_nuevo, columnas_numericas_con_precio + columnas_categoricas)
print(f'Error mediano: {num(error_nuevo, 1)} libras, contra {num(error_base, 1)} de antes.')

Error mediano: 43,8 libras, contra 71,2 de antes.


## Ahora la parte que se mira

La cifra bajó, y eso está bien. Pero **no es lo que se les pide justificar**.

Escriban, en una o dos frases:

- **de dónde sale** el valor de esta variable, o sea qué cuenta exactamente;
- **a qué hecho del negocio** corresponde, o sea qué decisión o comportamiento real refleja.

Sin mencionar cuánto bajó el error.

In [28]:
# escriban su justificación acá, entre las comillas
justificacion = """

"""
print(justificacion.strip() or 'Todavía sin escribir.')

Todavía sin escribir.


## Y si les queda tiempo

Prueben **otra** variable propia y compárenla. Dos ideas del archivo, las dos defendibles desde el
negocio:

- las **unidades por línea** de la factura, que distingue una compra al detalle de una al por mayor;
- la **hora** de la factura, que sale de `fecha_factura` igual que el mes.

La pregunta es la misma: qué cuenta, y qué hecho del negocio refleja.

## Atribución de datos

- **Creador:** Chen, D. (2012)
- **Fuente:** [UCI Machine Learning Repository — *Online Retail II*, dataset 502](https://archive.ics.uci.edu/dataset/502/online+retail+ii)
- **Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)
- **Modificación:** obra derivada en dos pasos. Las ocho columnas originales se renombraron al español
  en `snake_case`, sin traducir los valores; después se aplicaron las siete decisiones de limpieza de la
  clase 4, que eliminan filas repetidas y agregan seis columnas.